# Spatial Data Models

**A spatial data model** is a way of describing real-world objects in digital form, following a defined set of structures and rules.

GIS works with two of them: vector and raster.

**1. Vector data model**

Features are described by coordinates and drawn as points, lines and polygons. This suits discrete things with clear boundaries: buildings, roads, administrative units.

**2. Raster data model**

Space becomes a regular grid of cells, each holding a value. This suits continuous phenomena: terrain elevation, temperature, population density.


> In this section, we will explore the Python libraries and classes used for working with vector and raster spatial data


## 1. Vector Data Format


### 1.0. Importing Libraries


In [ ]:
from shapely.geometry import Point, LineString, Polygon
import geopandas as gpd

- [**Shapely**](https://shapely.readthedocs.io/) (`shapely.geometry`) is a Python library for working with vector geometry. You use it to create points, lines and polygons, and to run the standard geometric operations on them.

- [**GeoPandas**](https://geopandas.org/) (`geopandas`) is a Python library that extends [**pandas**](https://pandas.pydata.org/) – a popular library for working with tabular data in Python – to support geospatial data. With it you can load, process and analyse spatial datasets in a range of formats.


### 1.1. What Is Vector Data


The vector model describes features through their coordinates.

There are three main geometry types:

- **points** – represented by two (x, y) or three (x, y, z) coordinates.
- **lines** – an ordered sequence of coordinate pairs.
- **polygons** – a closed sequence of coordinate pairs where the first and last points coincide. You rarely write that repeated point yourself: Shapely closes the ring for you, so the four corners passed below come back as five coordinates.

Each geometry carries a set of attributes – the descriptive characteristics of the feature, held in tabular form.

Compound geometries are supported too (**MultiPoint, MultiLineString, MultiPolygon**): several geometries of the same type sharing one set of attributes. A land use layer is the usual place to meet one – a single area of woodland split by a road is one feature made of two polygons. We check a layer for them in [Exploring a Dataset](spData_3.ipynb).


### 1.2. Geometry: Shapely

We will use the **Shapely** classes to create a point, a line and a polygon. The point is the Wiener Staatsoper; the line and the polygon are invented – a few coordinates typed by hand next to it – because what matters here is the mechanics, not the geography.

Each cell below does two things. `print()` writes the geometry out as **WKT** (Well-Known Text), the standard way of expressing geometry as a plain string – you will meet it again in the third module, reading a file that stores its coordinates in a text column. The bare name on the last line lets Jupyter draw the shape itself.


In [ ]:
# point
point = Point(16.3691, 48.2029)
print(f"Geometry: {point}")
point

In [ ]:
# line
line = LineString([(16.3680, 48.2035), (16.3700, 48.2035), (16.3700, 48.2020)])
print(f"Geometry: {line}")

# object properties
print(f"Length: {line.length}")

line

In [ ]:
# polygon
polygon = Polygon([(16.3680, 48.2035), (16.3700, 48.2035), (16.3700, 48.2020), (16.3680, 48.2020)])
print(f"Geometry: {polygon}")

# object properties
print(f"Area: {polygon.area}")
print(f"Length: {polygon.length}")

polygon

Note that `length` on a polygon is its **perimeter** – the same attribute name means one thing for a line and another for a closed ring.

Note too that `length` and `area` are calculated in the units of the coordinates themselves. Here the coordinates are given in degrees of latitude and longitude, so the values above are expressed in degrees and square degrees – not in metres and square metres. Measuring real distances and areas requires a projected coordinate system, which we will cover in the second module.


At this stage, though, we have geometry and nothing else. That is not yet spatial data in the full sense: there are no attributes and no coordinate system.


### 1.3. Spatial Dataset: GeoPandas


A spatial dataset in Python is represented by the class `GeoDataFrame` from the **geopandas** library.

To create one, we need to combine geometry and attribute data into a unified tabular structure, and specify a coordinate reference system (CRS).

Now we build a `GeoDataFrame` from the `point` geometry defined above, adding attributes and a CRS.


In [ ]:
gdf = gpd.GeoDataFrame(
    {"name": ["Wiener Staatsoper"]},  # attributes
    geometry=[point],                # geometry
    crs="EPSG:4326"                  # coordinate system
)

The `crs` parameter defines the **coordinate reference system** – the rule that ties coordinates to real positions on Earth. Here we use the WGS84 geographic coordinate system with the code `EPSG:4326`, in which coordinates are expressed in degrees. We will discuss coordinate systems and how to choose them in more detail in the second module.


The result:


In [ ]:
gdf

At first glance, it looks almost identical to a regular `DataFrame`. However, the `geometry` column is not just another field – it enables spatial operations.


Using the `explore()` method of the `GeoDataFrame` class, we can put the feature on an interactive map and check that it is correctly georeferenced. This method relies on the `folium` and `mapclassify` libraries, which must be installed beforehand:


In [ ]:
gdf.explore(tiles='cartodbpositron')

The point lands where it should – the feature is correctly georeferenced, and we have built a spatial dataset from bare coordinates.


The same approach works for lines and polygons – any Shapely geometry can be passed to the `geometry` parameter together with its attributes and CRS:


In [ ]:
gdf_shapes = gpd.GeoDataFrame(
    {"name": ["Route", "Block"]},  # attributes
    geometry=[line, polygon],      # geometry
    crs="EPSG:4326"                # coordinate system
)

gdf_shapes

And, as with the point, we can check on a map that they are where we intended:


In [ ]:
gdf_shapes.explore(tiles="cartodbpositron")

## 2. Raster Data Format


The raster format differs from the vector model in several ways. Here we introduce it briefly; Module 5 returns to raster data in detail.


### 2.0. Importing Libraries


In [ ]:
import numpy as np
import rasterio
import rasterio.plot

- [**NumPy**](https://numpy.org/) (`numpy`) – a library for working with multidimensional arrays. In the context of raster data, it is used to store and process cell values, since a raster is essentially a numerical array.

- [**rasterio**](https://rasterio.readthedocs.io/) (`rasterio`) – a Python library for working with raster spatial data. It provides tools for reading, writing, and analysing raster files (e.g., GeoTIFF), as well as for handling coordinate systems, resolution, and spatial referencing.


### 2.1. What Is Raster Data


The raster model represents space as a regular grid of cells (pixels), each with a fixed position and size.

Key raster characteristics:

- **cell (pixel)** is the smallest raster element, containing a numeric value;
- **resolution** is the spatial size of a cell;
- **raster dimensions** is the number of rows and columns (width and height).

Unlike vector data, a raster does not store individual features and their attributes. Instead, each cell is assigned a value for some indicator. This format is well suited for representing continuous phenomena such as elevation, temperature, or population density.


### 2.2. Raster Data Structure


Raster data is stored as an array of values with associated spatial referencing, coordinate system, cell size (resolution), and other characteristics.


We will build one from scratch, in the same order as the `GeoDataFrame` in the first half: the raw material first, then the thing that makes it spatial.

The raw material is a plain array of numbers. The values below are invented – a simple ramp – because at this point the structure is what matters, not what is being measured.


In [ ]:
# the values: a plain NumPy array, with nothing spatial about it yet
raster_data = np.array([
    [1, 2, 3, 4, 5],
    [2, 3, 4, 5, 6],
    [3, 4, 5, 6, 7],
    [4, 5, 6, 7, 8],
    [5, 6, 7, 8, 9]
], dtype=rasterio.float32)

raster_data

At this point we have numbers and nothing else – the same position we were in with the bare Shapely geometry above. The array knows that it has five rows and five columns, and nothing at all about where on Earth those rows and columns lie.

What ties it to the Earth is the **transform**: the rule that turns a row and column index into a coordinate. `from_origin()` builds one from the coordinates of the **upper-left corner** and the size of a cell.


In [ ]:
transform = rasterio.transform.from_origin(16.35, 48.22, 0.01, 0.01)

transform

The transform is an **affine matrix**, and it repays reading rather than being treated as a black box:

```
| 0.01, 0.00, 16.35|
| 0.00,-0.01, 48.22|
```

The right-hand column is the upper-left corner. The `0.01` is the width of a cell in degrees of longitude. And the cell height is **negative** – because raster rows are counted downwards from the top, the way an image is stored, while latitude increases upwards. That minus sign is the entire reason a raster is anchored at its upper-left corner rather than its lower-left.

The corner and cell size chosen here put the grid over **the same corner of Vienna** as the point and the polygon in the first half of this section: the same place, described by the other model.

With both pieces in hand, any cell of the array can be asked where it is:


In [ ]:
for row, col in [(0, 0), (0, 4), (4, 0)]:
    x, y = rasterio.transform.xy(transform, row, col)
    print(f"cell [row {row}, col {col}] -> {x:.4f}, {y:.4f}")

Row 0 comes back at latitude 48.215 and row 4 at 48.175: moving down the array moves south across the map, exactly as the minus sign in the transform promised.

**Array plus transform is the raster data model**, in the same way that geometry plus CRS was the vector one.


And the raster itself:


In [ ]:
rasterio.plot.show(raster_data, transform=transform)

Two things are worth reading off the axes. The grid sits over the same coordinates as the vector examples above – and each of its cells measures about **740 by 1,100 metres**, while the polygon we drew earlier is roughly 150 by 170. A single cell of this raster is larger than that entire feature.

That is the trade-off between the two models, in one number. The vector model holds a block as a block; a raster holds whatever fits its resolution, and everything smaller is averaged away. Hence the rule of thumb from the top of this section: a raster of population density at this cell size is perfectly reasonable, a raster of opera houses is not.

We have examined the basic structure of raster data – its representation as a numerical matrix and its spatial referencing parameters.

In Module 5, we will explore raster data in depth, including its characteristics, data types, reading methods, and analysis techniques.


## Summary


In this section, we explored the two primary spatial data models – **vector** and **raster**.

- The **vector model** describes individual features: geometry, plus a coordinate reference system that ties it to the Earth, plus attributes.
- The **raster model** describes a surface: an array of values, plus a transform that ties its rows and columns to the Earth.

Both follow the same shape – raw material, and a rule that makes it spatial. What separates them is the resolution question we ran into above: the vector model can hold a feature of any size, while a raster can hold nothing smaller than one cell.

We reviewed the Python libraries used for working with each format.

In the following sections, we will explore vector data in more detail, and towards the end of the course, we will also work with raster data.


## References


Campbell, J. B., & Shin, M. (2011). Essentials of geographic information systems. Saylor Academy. https://saylordotorg.github.io/text_essentials-of-geographic-information-systems/
